In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import math

In [2]:
# -------------------------------------------------------------------
# 1. Parameters and file paths
# -------------------------------------------------------------------

# File paths
SUPPLY_PLAN_FILE = "../data/CUL October 2025 Supply Plan.xlsx"
REQUESTS_FILE2 = "../data/Ride Requests_2025-11-11-part-2.csv"
REQUESTS_FILE1 = "../data/Ride Requests_2025-11-11-part-1.csv"
RIDES_FILE = "../data/rides.csv"

# Time window for estimating λ and μ
OCT_START = pd.Timestamp("2025-10-01")
OCT_END = pd.Timestamp("2025-11-01")  # exclusive upper bound

# Driver cost assumptions
HOURLY_WAGE_8H = 20.0  # $/hour for 8-hour shift drivers
HOURLY_WAGE_4H = 25.0  # $/hour for 4-hour shift drivers

# Range of servers (drivers) to evaluate
MIN_DRIVERS = 5
MAX_DRIVERS = 30

# -------------------------------------------------------------------
# 2. Load data
# -------------------------------------------------------------------

supply_plan = pd.read_excel(SUPPLY_PLAN_FILE)
requests1 = pd.read_csv(REQUESTS_FILE1)
requests2 = pd.read_csv(REQUESTS_FILE2)
request = pd.concat([requests1, requests2], ignore_index=True)
rides = pd.read_csv(RIDES_FILE)


# -------------------------------------------------------------------
# 3. Helpers
# -------------------------------------------------------------------
def weekday_sorter(df):
    weekday_order = [
        "Monday",
        "Tuesday",
        "Wednesday",
        "Thursday",
        "Friday",
        "Saturday",
        "Sunday",
    ]

    df["weekday"] = pd.Categorical(
        df["weekday"], categories=weekday_order, ordered=True
    )
    df = df.sort_values("weekday")
    return df


def parse_datetime_column(df, col_name):
    return pd.to_datetime(df[col_name], errors="coerce", infer_datetime_format=True)

In [3]:
# -------------------------------------------------------------------
# 8 bis. Estimate λ using 'Request Creation Time' column (to include not only completed rides)
# -------------------------------------------------------------------

# Parse 'Request Creation Time'
request["Request Creation Time"] = parse_datetime_column(
    request, "Request Creation Time"
)
# Filter to October 2025
oct_mask_req = (request["Request Creation Time"] >= OCT_START) & (
    request["Request Creation Time"] < OCT_END
)
request_oct = request.loc[oct_mask_req].copy()

# Convert 'Request Creation Time' to weekday and hour
request_oct["weekday"] = request_oct["Request Creation Time"].dt.day_name()
request_oct["hour"] = request_oct["Request Creation Time"].dt.hour
# Count requests per weekday × hour over the whole month
lambda_counts_req = (
    request_oct.groupby(["weekday", "hour"])["Request Creation Time"]
    .count()
    .reset_index()
    .rename(columns={"Request Creation Time": "num_requests"})
)
lambda_counts_req = weekday_sorter(lambda_counts_req)
print("\nRide requests counts per weekday × hour:")
print(lambda_counts_req)

/var/folders/j5/g36rlrm17vv157kr1c8_ndyw0000gn/T/ipykernel_38469/493454445.py:56: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  return pd.to_datetime(df[col_name], errors="coerce", infer_datetime_format=True)



Ride requests counts per weekday × hour:
   weekday  hour  num_requests
19  Monday    23           438
18  Monday    22           551
17  Monday    21           650
16  Monday    20           731
15  Monday    19           620
..     ...   ...           ...
35  Sunday    19           473
36  Sunday    20           589
37  Sunday    21           496
39  Sunday    23           405
34  Sunday    18           446

[70 rows x 3 columns]


In [4]:
# -------------------------------------------------------------------
# 10. M/M/m performance formulas and simulation with extra drivers
# -------------------------------------------------------------------


def mm_m_waiting_time(lambda_rate, mu_rate, m):
    """
    Compute Wq (average waiting time in queue) and W (total sojourn time)
    for an M/M/m system, using the standard Erlang-C formula.

    Parameters
    ----------
    lambda_rate : float
        Arrival rate (per day x hour).
    mu_rate : float
        Service rate per server (per day x hour).
    m : int
        Number of servers.

    Returns
    -------
    Wq : float
        Average waiting time in queue (in hours). np.nan if system is unstable.
    W : float
        Average time in the system (waiting + service) in hours.
    """

    # No arrivals: no queue, only service time
    if lambda_rate <= 0:
        Wq = 0.0
        W = 1.0 / mu_rate
        return Wq, W

    # Utilization ρ
    rho = lambda_rate / (m * mu_rate)

    # If ρ ≥ 1, the system is unstable
    if rho >= 1.0:
        return np.nan, np.nan

    # mρ = λ/μ
    m_rho = m * rho

    # Compute P0 (probability of zero customers in the system)
    sum_terms = 0.0
    for k in range(m):
        sum_terms += (m_rho**k) / math.factorial(k)

    # Last term represents the geometric tail (states n ≥ m)
    last_term = (m_rho**m) / (math.factorial(m) * (1 - rho))

    denominator = sum_terms + last_term
    P0 = 1.0 / denominator

    # Erlang-C: probability that an arriving customer must wait
    P_wait = last_term * P0

    # Average number in queue (closed form)
    Lq = P_wait * (rho / (1 - rho))

    # Average waiting time in queue
    Wq = Lq / lambda_rate

    # Total time in system (waiting + service)
    W = Wq + 1.0 / mu_rate

    return Wq, W

In [ ]:
# -------------------------------------------------------------------
# 5. Estimate μ (service rate) by weekday × hour
# -------------------------------------------------------------------

# Extract weekday + hour from pickup time
rides_oct["weekday"] = rides_oct["Actual Pickup Time"].dt.day_name()
rides_oct["hour"] = rides_oct["Actual Pickup Time"].dt.hour

# # Keep rides that are within the same hour
# rides_oct = rides_oct[
#     rides_oct["Actual Dropoff Time"].dt.floor("H")
#     == rides_oct["Actual Pickup Time"].dt.floor("H")
# ].copy()

# Compute ride durations in hours
ride_durations_hours = (
    rides_oct["Actual Dropoff Time"] - rides_oct["Actual Pickup Time"]
).dt.total_seconds() / 3600.0

rides_oct["duration_hours"] = ride_durations_hours

# Keep only positive durations
rides_oct = rides_oct[rides_oct["duration_hours"] > 0].copy()

# Group by weekday × hour
mu_df = (
    rides_oct.groupby(["weekday", "hour"])["duration_hours"]
    .mean()
    .reset_index()
    .rename(columns={"duration_hours": "mean_service_time_hours"})
)

# Convert mean duration to μ (service rate per hour per driver)
mu_df["mu_per_hour"] = 1.0 / mu_df["mean_service_time_hours"]

# Sort weekdays properly
mu_df = weekday_sorter(mu_df)
mu_df.sort_values(by=["weekday", "hour"], inplace=True)

print("\nWeekday × hour model (μ):")
print(mu_df)
# -------------------------------------------------------------------
# 6. Extract staffing by weekday x hour from the CUL supply plan
# -------------------------------------------------------------------

# Ensure Date is datetime
supply_plan["Date"] = pd.to_datetime(supply_plan["Date"])

# Map CUL columns to hour-of-day (0-23)
hour_column_map = {
    "4p-5p": 16,
    "5p-6p": 17,
    "6p-7p": 18,
    "7p-8p": 19,
    "8p-9p": 20,
    "9p-10p": 21,
    "10p-11p": 22,
    "11p-12a": 23,
    "12a-1a": 0,
    "1a-2a": 1,
    "2a-3a": 2,
}

hour_cols = list(hour_column_map.keys())

# Melt to long format: one row per (date, hour-block)
supply_long = supply_plan.melt(
    id_vars=["Date"], value_vars=hour_cols, var_name="time_block", value_name="drivers"
)

# Remove rows without drivers info
supply_long = supply_long.dropna(subset=["drivers"])

# Add weekday and numeric hour
supply_long["weekday"] = supply_long["Date"].dt.day_name()
supply_long["hour"] = supply_long["time_block"].map(hour_column_map)

# Average staffing by weekday × hour over October
weekday_hour_staffing = (
    supply_long.groupby(["weekday", "hour"])["drivers"]
    .mean()
    .reset_index()
    .rename(columns={"drivers": "avg_drivers"})
)
weekday_hour_staffing["avg_drivers"] = weekday_hour_staffing["avg_drivers"].round(0)
weekday_hour_staffing = weekday_sorter(weekday_hour_staffing)
weekday_hour_staffing.sort_values(by=["weekday", "hour"], inplace=True)

print("\nAverage staffing per weekday × hour:")
print(weekday_hour_staffing)

# Also compute an overall "current_drivers" from the peak column (for the vertical line)
avg_peak_drivers = supply_plan["Drivers @ Peak"].mean()
current_drivers = int(round(avg_peak_drivers))
print(
    f"\nApproximate current number of drivers (avg peak over October): {current_drivers}"
)


def combine_staffing_requests(
    weekday_hour_staffing, lambda_counts_req, days_per_weekday, mu_df
):
    # Merge staffing (weekday, hour) with observed request counts (weekday, hour)
    weekday_hour_model_req = weekday_hour_staffing.merge(
        lambda_counts_req, on=["weekday", "hour"], how="left"
    )

    # Merge in the number of days per weekday (for exposure)
    weekday_hour_model_req = weekday_hour_model_req.merge(
        days_per_weekday, on="weekday", how="left"
    )

    # If no requests observed for a (weekday, hour), set num_requests = 0
    weekday_hour_model_req["num_requests"] = weekday_hour_model_req[
        "num_requests"
    ].fillna(0)

    # Arrival rate per hour = total requests / number of days of that weekday
    weekday_hour_model_req["lambda_per_hour"] = (
        weekday_hour_model_req["num_requests"] / weekday_hour_model_req["num_days"]
    )

    # Merge with service rate μ
    weekday_hour_model_req = weekday_hour_model_req.merge(
        mu_df[["weekday", "hour", "mu_per_hour"]], on=["weekday", "hour"], how="left"
    )

    return weekday_hour_model_req


# For requests data
weekday_hour_model_req = combine_staffing_requests(
    weekday_hour_staffing, lambda_counts_req, days_per_weekday, mu_df
)


def simul_waittimes_m_req(day, hour, M):
    """
    Returns a list of average waiting times in minutes for an M/M/m system for a given day and hour, varying the number of drivers from base_drivers to base_drivers + max_extra.
    Parameters
    ----------
    day : str
        Day of the week (e.g., "Monday").
    hour : int
        Hour of the day (0-23).
    base_drivers : int
        Base number of drivers.
    max_extra : int
        Maximum number of extra drivers to simulate.
    Returns
    -------
    list of float
        Average waiting times in minutes for each number of drivers from
        base_drivers to base_drivers + max_extra.
    """
    row = weekday_hour_model_req[
        (weekday_hour_model_req["weekday"] == day)
        & (weekday_hour_model_req["hour"] == hour)
    ].iloc[0]

    lam = row["lambda_per_hour"]
    mu = row["mu_per_hour"]
    results = []
    for m in range(M):
        Wq, W = mm_m_waiting_time(lam, mu, m)

        if Wq is None or math.isnan(Wq):
            results.append(None)
        else:
            results.append(round(Wq * 60, 2))

    return results


Average staffing per weekday × hour:
   weekday  hour  avg_drivers
11  Monday     0         10.0
12  Monday     1          6.0
13  Monday     2          4.0
14  Monday    16          0.0
15  Monday    17          0.0
..     ...   ...          ...
39  Sunday    19          8.0
40  Sunday    20         14.0
41  Sunday    21         14.0
42  Sunday    22         14.0
43  Sunday    23         12.0

[77 rows x 3 columns]

Approximate current number of drivers (avg peak over October): 16


NameError: name 'days_per_weekday' is not defined